# ATENEA
### Plataforma de detección de riesgo académico

**Proyecto Integrador** · ODS 4 (Educación de calidad)

Stack: PHP · MySQL/MariaDB · JavaScript · Python

---
*Este notebook acompaña la exposición oral del proyecto: explica el problema, la arquitectura y el motor de análisis con los fragmentos de código más representativos.*


## 1. Problema y objetivo

Muchos estudiantes abandonan o reprueban sin que nadie lo detecte a tiempo. Las señales de alerta (bajas calificaciones, poca asistencia, estrés, malos hábitos de estudio) suelen existir, pero **nadie las junta ni las traduce en una alerta clara**.

**Atenea** ataca ese problema: convierte una encuesta académica en un **puntaje de riesgo** (0–100) y clasifica al estudiante en **Bajo / Medio / Alto**, para que tanto el alumno como la institución puedan actuar antes de que el problema crezca.

Objetivo concreto: registrar al estudiante, levantar su encuesta, calcular su riesgo automáticamente y mostrárselo en un dashboard, con un panel de administración para dar seguimiento global.


## 2. Arquitectura general

El proyecto sigue un esquema clásico de aplicación web con **PHP + MySQL**, más un módulo independiente en **Python** que reproduce el mismo análisis fuera del navegador.

```
Estudiante
   │
   ▼
Landing Page  →  Registro / Login  →  Encuesta (form.php)
                                            │
                                            ▼
                                   Base de datos (MySQL)
                                   usuarios · encuestas · resultados
                                            │
                        ┌───────────────────┴───────────────────┐
                        ▼                                       ▼
              Motor de riesgo en PHP                  Motor de riesgo en Python
              (analytics.php → dashboard)              (risk_analysis.py → JSON)
```

Puntos clave para la expo:
- El **mismo algoritmo de riesgo está implementado dos veces**, una en PHP (para que el dashboard lo muestre en vivo) y otra en Python (para análisis externo, reportes o un futuro modelo de ML).
- Todo el flujo de usuario pasa por una sola base de datos relacional, lo que mantiene consistencia entre lo que ve el estudiante y lo que ve el administrador.


In [ ]:
# conexion.php — punto único de conexión a la base de datos
# Todos los demás archivos PHP lo incluyen con require/include

"""
$host = "localhost";
$user = "root";
$pass = "";
$db   = "atenea";

$conn = new mysqli($host, $user, $pass, $db);

if ($conn->connect_error) {
    die("Error de conexión: " . $conn->connect_error);
}

$conn->set_charset("utf8mb4");
"""
print("Archivo: PHP/conexion.php — conexión centralizada a MySQL (mysqli)")


## 3. Base de datos

Siete tablas en MySQL/MariaDB. Las tres que sostienen el flujo principal:

| Tabla | Para qué sirve |
|---|---|
| `usuarios` | identidad, correo, carrera, rol (`usuario`/`admin`), estado de la cuenta |
| `encuestas` | respuestas académicas y de hábitos (promedio, asistencia, sueño, estrés…) |
| `resultados` | puntuación y nivel de riesgo ya calculados |

Las otras cuatro (`recomendaciones`, `sesiones`, `password_resets`, `admin_historial`) dan soporte a funciones secundarias: plan de mejora, historial de accesos, recuperación de contraseña y auditoría de administrador.


In [ ]:
# Estructura real de la tabla `encuestas` (DATABASE/atenea.sql)
# Aquí se guarda cada respuesta del cuestionario académico

"""sql
CREATE TABLE encuestas (
  id                     INT PRIMARY KEY AUTO_INCREMENT,
  usuario_id             INT,
  promedio               DECIMAL(4,2),
  materias_reprobadas    INT,
  asistencia             INT,
  horas_estudio          DECIMAL(3,1),
  horas_sueno            DECIMAL(3,1),
  uso_redes              DECIMAL(3,1),
  nivel_estres           TINYINT,
  desmotivacion          TINYINT,
  administracion_tiempo  TINYINT,
  entrega_tareas         TINYINT,
  acceso_internet        TINYINT(1),
  espacio_estudio        TINYINT(1),
  trabaja                TINYINT(1),
  fecha_encuesta         TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
print("Cada fila de 'encuestas' es la materia prima que alimenta al motor de riesgo.")


## 4. El motor de riesgo: cómo se calcula el puntaje

Esta es la parte central de Atenea. No es un simple promedio: es una **suma ponderada** de tres bloques, donde lo académico pesa más que el resto.

| Bloque | Variables | Peso máximo aproximado |
|---|---|---|
| Académico | promedio, materias reprobadas, asistencia | 76 pts |
| Hábitos | horas de estudio, horas de sueño, uso de redes | 21 pts |
| Contexto | estrés, desmotivación, organización del tiempo, entrega de tareas, internet, espacio de estudio, si trabaja | el resto |

El resultado se limita a 100 puntos y luego se traduce a una categoría:

- **Bajo**: puntaje < 32
- **Medio**: 32 ≤ puntaje < 65
- **Alto**: puntaje ≥ 65

La lógica está **duplicada intencionalmente** en PHP (para el dashboard web) y en Python (para análisis externo). A continuación el bloque más representativo de cada uno.


In [ ]:
# PYTHON/risk_analysis.py — función principal del motor de riesgo
# (idéntica en espíritu a la versión PHP de analytics.php)
# Se muestra completa: cada bloque (académico / hábitos / contexto) suma puntos.

def risk_score(row: dict) -> float:
    score = 0.0

    promedio = float(row.get("promedio") or 0)
    reprobadas = int(row.get("materias_reprobadas") or 0)
    asistencia = int(row.get("asistencia") or 0)
    estudio = float(row.get("horas_estudio") or 0)
    sueno = float(row.get("horas_sueno") or 0)
    redes = float(row.get("uso_redes") or 0)
    estres = int(row.get("nivel_estres") or 0)
    desmotivacion = int(row.get("desmotivacion") or 0)
    tiempo = int(row.get("administracion_tiempo") or 0)
    entrega = int(row.get("entrega_tareas") or 3)

    # --- Bloque académico (pesa más) ---
    score += (
        30 if promedio < 6 else
        22 if promedio < 7 else
        14 if promedio < 7.5 else
        8  if promedio < 8 else
        4  if promedio < 8.5 else
        0
    )

    if reprobadas == 1: score += 5
    elif reprobadas == 2: score += 10
    elif reprobadas == 3: score += 15
    elif reprobadas >= 4: score += 20

    score += (
        26 if asistencia < 60 else
        20 if asistencia < 75 else
        13 if asistencia < 85 else
        7  if asistencia < 90 else
        3  if asistencia < 95 else
        0
    )

    # --- Bloque de hábitos ---
    score += (
        8 if estudio < 1 else 6 if estudio < 2 else
        4 if estudio < 3 else 2 if estudio < 5 else 0
    )
    score += (
        9 if sueno < 5 else 6 if sueno < 5.5 else
        3 if sueno < 6.5 else 0
    )
    score += (
        4 if redes > 6 else 2 if redes > 4 else
        1 if redes > 2 else 0
    )

    # --- Bloque de contexto y bienestar ---
    score += (
        12 if estres >= 10 else 10 if estres >= 8 else
        6 if estres >= 6 else 3 if estres >= 4 else 0
    )
    score += (
        5 if desmotivacion >= 4 else 3 if desmotivacion == 3 else
        1 if desmotivacion == 2 else 0
    )
    score += (
        6 if tiempo <= 1 else 4 if tiempo == 2 else
        2 if tiempo == 3 else 0
    )
    score += 4 if entrega <= 1 else 2 if entrega == 2 else 0

    return round(min(100, score), 2)


def risk_level(score: float) -> str:
    if score >= 65:
        return "Alto"
    if score >= 32:
        return "Medio"
    return "Bajo"


In [ ]:
# Ejemplo con un caso real de la base de datos (tabla encuestas, fila id=5)
# promedio=8.9, asistencia=10% (muy baja) pero buenos hábitos en lo demás

estudiante_ejemplo = {
    "promedio": 8.9,
    "materias_reprobadas": 0,
    "asistencia": 10,
    "horas_estudio": 4.0,
    "horas_sueno": 5.0,
    "uso_redes": 4.0,
    "nivel_estres": 8,
    "desmotivacion": 3,
    "administracion_tiempo": 3,
    "entrega_tareas": 4,
}

puntaje = risk_score(estudiante_ejemplo)
nivel = risk_level(puntaje)

print(f"Puntaje de riesgo: {puntaje}")
print(f"Nivel de riesgo:   {nivel}")

# Nota para la expo: aunque el promedio es alto (8.9), la asistencia tan baja
# (10%) por sí sola ya suma 26 de los ~32 puntos necesarios para "Medio".
# Así se ve el peso real de cada bloque, no solo de las calificaciones.


**Por qué existe la versión en PHP además de la de Python**

- La versión en **PHP** (`analytics.php`) corre dentro del dashboard y muestra el resultado al estudiante en tiempo real, justo después de llenar la encuesta.
- La versión en **Python** (`risk_analysis.py`) se conecta directamente a la base de datos con `PyMySQL`, recorre a todos los estudiantes y genera un **reporte en JSON**: riesgo promedio general, distribución de niveles, top de mayor riesgo y mejores promedios. Es la base para futuros reportes automáticos o un modelo de Machine Learning.


In [ ]:
# Fragmento de risk_analysis.py: cómo se arma el reporte final (JSON)
# Esto se ejecuta fuera del navegador, por consola

"""python
output = {
    "total_estudiantes": len(enriched),
    "riesgo_promedio": round(mean(scores), 2) if scores else 0,
    "distribucion_riesgo": distribution,          # {"Bajo": x, "Medio": y, "Alto": z}
    "mayor_riesgo": sorted(enriched, key=lambda i: i["puntuacion_riesgo"],
                           reverse=True)[:limit],
    "mejor_promedio": sorted(enriched, key=lambda i: i["promedio"],
                              reverse=True)[:limit],
}

print(json.dumps(output, ensure_ascii=False, indent=2))
"""
print("Salida: un JSON listo para alimentar reportes, dashboards externos o un futuro modelo de ML.")


## 5. Flujo de usuario y dashboard

1. El estudiante llega a la **Landing Page** (con estadísticas reales tomadas en vivo de la BD: total de estudiantes, encuestas, promedio global, perfiles de riesgo alto).
2. Se registra o inicia sesión (`password_hash` / `password_verify`, consultas preparadas).
3. Completa la **encuesta académica** (formulario multi-step).
4. El sistema calcula su riesgo y lo manda al **dashboard**, que carga vistas dinámicamente desde `VIEWS/` según la sección: inicio, ranking, perfil, análisis, plan de acción, comunidad y actividad.
5. Si quien entra es un **administrador**, accede a un panel separado con control total: editar usuarios, cambiar roles, bloquear cuentas y consultar el historial de acciones (auditoría).

Seguridad ya implementada: contraseñas hasheadas, consultas preparadas para evitar inyección SQL, control de acceso por rol, bloqueo de cuentas y tokens de un solo uso para recuperar contraseña.


## 6. Estado actual y siguiente paso

**Ya funciona:** landing dinámica, registro + encuesta, login, dashboard de estudiante, dashboard de administrador, recuperación de contraseña, motor de riesgo (PHP y Python) e historial de sesiones.

**Lo que sigue:** una capa de asistente con IA (ya hay una vista preparada en `VIEWS/assistant.php`) para dar consejos personalizados con base en el resultado de la encuesta y el historial del estudiante, usando un modelo local (`ollama` con `qwen3:8b`).

En una frase: Atenea ya convierte datos académicos crudos en una señal de alerta clara y accionable, con una base sólida para crecer hacia análisis predictivo más avanzado.
